# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
%load_ext dotenv
%dotenv 

In [2]:
import dask.dataframe as dd

c:\Users\Joanne\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
# LOAD
import os
from glob import glob
# loads the environment variable PRICE_DATA
PRICE_DATA = os.getenv("PRICE_DATA")
# use glob to find the path of all parquet files in the directory PRICE_DATA
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
# use Dask to store list of all parquet file paths in parquet_files and reads files into Dask DataFrame
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [28]:
# TRANSFORM
# add lags for variables Close and Adj_Close
dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = x['Close'].shift(1),
        Adj_Close_lag_1 = x['Adj Close'].shift(1)
    )
)
# add returns based on Close
dd_feat = dd_feat.assign(
    returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
)
# add the hi_lo_range and assign the result to dd_feat
dd_feat = dd_feat.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(
        hi_lo_range = x['High'] - x['Low']
    ),
)

C:\Users\Joanne\AppData\Local\Temp\ipykernel_16292\1329332534.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
C:\Users\Joanne\AppData\Local\Temp\ipykernel_16292\1329332534.py:14: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_feat.groupby('ticker', group_keys=False).apply(


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [29]:
# Using .compute to convert Dask dataframe to pandas dataframe
import pandas as pd
df_feat = dd_feat.compute()

# add a new feature containing moving average of returns using a window of 10 days
df_feat = df_feat.assign(
    moving_average = lambda x: x['returns'].rolling(10).mean()
)

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

> No, it was not strictly necessary to convert from Dask to pandas to calculate the moving average return. ChatGPT tells me that Dask supports rolling window operations like .rolling () and .mean() directly on Dask dataframes, so I could have computed the moving average in Dask before converting to pandas (just like I computed returns and hi_lo_range). 

> I think it would have been better to do it in Dask. Dask is designed to handle large datasets that don't fit into memory by processing data in chunks and parallelizing computations. Converting to pandas with the .compute() function requires loading the entire dataset into memory, whereas if I had done it in Dask I would have leveraged Dask's out-of-memory capabilities, keeping memory usage more manageable. This was evidenced by the quick run times of the other calculations I computed in Dask, vs. it took almost 20 seconds to compute it in pandas. 

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.